In [1]:
import torch
import torch.nn.functional as F

# -------------------------------------------------------
# 1. Load GloVe embeddings from file
# -------------------------------------------------------

def load_glove(path, dim):
    embeddings = {}
    with open(path, 'r', encoding='utf8') as f:
        for line in f:
            parts = line.strip().split()
            if not parts:
                continue
            word = parts[0]
            vector_str = parts[1:]
            if len(vector_str) != dim:
                # Optionally, print a warning for debugging:
                print(f"Warning: Skipping word '{word}' due to dimension mismatch. Expected {dim}, got {len(vector_str)}.")
                continue
            try:
                vector = torch.tensor(list(map(float, vector_str)), dtype=torch.float)
                embeddings[word] = vector
            except ValueError:
                # Optionally, print a warning for debugging:
                print(f"Warning: Skipping word '{word}' due to non-float values in vector.")
                continue
    return embeddings

glove_path = "./glove.6B.300d.txt"   # <-- set your path here
embeddings = load_glove(glove_path, dim=300) # Changed dim to match the filename

# -------------------------------------------------------
# 2. Helper: find nearest neighbors
# -------------------------------------------------------

def most_similar(v, embeddings, top_k=5):
    words = []
    vecs = []

    for w, e in embeddings.items():
        words.append(w)
        vecs.append(e.unsqueeze(0))

    mat = torch.cat(vecs, dim=0)      # V × 300 matrix
    v = v / v.norm()                  # normalize
    mat_norm = mat / mat.norm(dim=1, keepdim=True)

    cosine_sim = torch.matmul(mat_norm, v)
    topk = torch.topk(cosine_sim.squeeze(), top_k)

    results = [(words[i], float(topk.values[j]))
               for j, i in enumerate(topk.indices)]
    return results

# -------------------------------------------------------
# 3. Compute king - man + woman
# -------------------------------------------------------

king = embeddings["king"]
man = embeddings["man"]
woman = embeddings["woman"]

result_vec = king - man + woman

# -------------------------------------------------------
# 4. Print nearest neighbors
# -------------------------------------------------------

neighbors = most_similar(result_vec, embeddings, top_k=10)
print("Top neighbors of (king - man + woman):")
for w, score in neighbors:
    print(f"{w:15s}  cosine={score:.4f}")


Top neighbors of (king - man + woman):
king             cosine=0.8066
queen            cosine=0.6896
monarch          cosine=0.5575
throne           cosine=0.5565
princess         cosine=0.5519
mother           cosine=0.5142
daughter         cosine=0.5133
kingdom          cosine=0.5025
prince           cosine=0.5018
elizabeth        cosine=0.4908


In [2]:
Colder = embeddings["colder"]
Cold = embeddings["cold"]
Hot = embeddings["hot"]

result_vec = Colder - Cold + Hot

# -------------------------------------------------------
# 4. Print nearest neighbors
# -------------------------------------------------------

neighbors = most_similar(result_vec, embeddings, top_k=10)
print("Top neighbors of (Colder - Cold + Hot):")
for w, score in neighbors:
    print(f"{w:15s}  cosine={score:.4f}")

Top neighbors of (Colder - Cold + Hot):
hotter           cosine=0.7156
colder           cosine=0.6509
hot              cosine=0.6181
drier            cosine=0.5834
cooler           cosine=0.5628
warmer           cosine=0.5200
hottest          cosine=0.4759
wetter           cosine=0.4677
humid            cosine=0.4401
climates         cosine=0.4184


In [3]:
num_words = len(embeddings)
embedding_dim = list(embeddings.values())[0].shape[0]

print(f"The GloVe embedding matrix has {num_words} words and each embedding is {embedding_dim} dimensions long.")

The GloVe embedding matrix has 400000 words and each embedding is 300 dimensions long.


In [4]:
# -------------------------------------------------------
# 5. Cosine similarity and dot-product for word pairs
# -------------------------------------------------------

def cosine_similarity(v1: torch.Tensor, v2: torch.Tensor) -> float:
    """Compute cosine similarity between two vectors."""
    v1_norm = v1 / v1.norm()
    v2_norm = v2 / v2.norm()
    return float(torch.dot(v1_norm, v2_norm))

def dot_product(v1: torch.Tensor, v2: torch.Tensor) -> float:
    """Compute dot product between two vectors."""
    return float(torch.dot(v1, v2))

# Define pairs of related words
related_pairs = [
    ("king", "queen"),
    ("cat", "dog"),
    ("happy", "joyful"),
    ("car", "vehicle"),
    ("doctor", "nurse"),
    ("sun", "moon"),
    ("coffee", "tea"),
    ("france", "paris"),
]

# Define pairs of unrelated words
unrelated_pairs = [
    ("king", "banana"),
    ("cat", "democracy"),
    ("happy", "refrigerator"),
    ("car", "philosophy"),
    ("doctor", "volcano"),
    ("sun", "keyboard"),
    ("coffee", "elephant"),
    ("france", "purple"),
]

def compute_similarities(pairs: list[tuple[str, str]], label: str):
    """Compute and display cosine similarity and dot product for word pairs."""
    print(f"\n{'='*60}")
    print(f"{label}")
    print(f"{'='*60}")
    print(f"{'Word 1':<15} {'Word 2':<15} {'Cosine Sim':>12} {'Dot Product':>14}")
    print(f"{'-'*60}")
    
    cosines = []
    dots = []
    
    for w1, w2 in pairs:
        if w1 not in embeddings or w2 not in embeddings:
            print(f"{w1:<15} {w2:<15} {'[word not found]':>26}")
            continue
        
        v1 = embeddings[w1]
        v2 = embeddings[w2]
        
        cos_sim = cosine_similarity(v1, v2)
        dot_prod = dot_product(v1, v2)
        
        cosines.append(cos_sim)
        dots.append(dot_prod)
        
        print(f"{w1:<15} {w2:<15} {cos_sim:>12.4f} {dot_prod:>14.4f}")
    
    if cosines:
        print(f"{'-'*60}")
        print(f"{'Average':<30} {sum(cosines)/len(cosines):>12.4f} {sum(dots)/len(dots):>14.4f}")

# Compute similarities for both related and unrelated pairs
compute_similarities(related_pairs, "RELATED WORD PAIRS")
compute_similarities(unrelated_pairs, "UNRELATED WORD PAIRS")


RELATED WORD PAIRS
Word 1          Word 2            Cosine Sim    Dot Product
------------------------------------------------------------
king            queen                 0.6336        30.7786
cat             dog                   0.6817        28.7780
happy           joyful                0.4751        15.1615
car             vehicle               0.7655        35.8615
doctor          nurse                 0.5860        23.5423
sun             moon                  0.4807        24.7203
coffee          tea                   0.6692        32.3421
france          paris                 0.6581        34.2710
------------------------------------------------------------
Average                              0.6187        28.1819

UNRELATED WORD PAIRS
Word 1          Word 2            Cosine Sim    Dot Product
------------------------------------------------------------
king            banana                0.0666         3.0614
cat             democracy             0.0007         0.0

In [5]:
# -------------------------------------------------------
# 6. Comparación de GloVe 50d, 100d, 300d
# -------------------------------------------------------

print("Cargando embeddings de diferentes dimensiones...")

# Cargar los tres modelos
glove_50d = load_glove("./glove.6B.50d.txt", dim=50)
print(f"✓ GloVe 50d cargado: {len(glove_50d)} palabras")

glove_100d = load_glove("./glove.6B.100d.txt", dim=100)
print(f"✓ GloVe 100d cargado: {len(glove_100d)} palabras")

glove_300d = load_glove("./glove.6B.300d.txt", dim=300)
print(f"✓ GloVe 300d cargado: {len(glove_300d)} palabras")

# Diccionario con todos los modelos
glove_models = {
    "50d": glove_50d,
    "100d": glove_100d,
    "300d": glove_300d,
}

Cargando embeddings de diferentes dimensiones...


✓ GloVe 50d cargado: 400000 palabras
✓ GloVe 100d cargado: 400000 palabras
✓ GloVe 300d cargado: 400000 palabras


In [6]:
# -------------------------------------------------------
# 7. Función para comparar métricas entre dimensiones
# -------------------------------------------------------

def compare_across_dimensions(
    pairs: list[tuple[str, str]], 
    models: dict[str, dict], 
    pair_type: str
) -> dict[str, dict[str, list[float]]]:
    """
    Compara cosine similarity y dot-product para pares de palabras
    a través de diferentes dimensiones de embeddings.
    """
    results = {dim: {"cosine": [], "dot": []} for dim in models.keys()}
    
    print(f"\n{'='*90}")
    print(f"  {pair_type}")
    print(f"{'='*90}")
    print(f"{'Palabra 1':<12} {'Palabra 2':<12} │ {'50d Cos':>8} {'50d Dot':>10} │ {'100d Cos':>8} {'100d Dot':>10} │ {'300d Cos':>8} {'300d Dot':>10}")
    print(f"{'-'*90}")
    
    for w1, w2 in pairs:
        row = f"{w1:<12} {w2:<12} │"
        valid = True
        
        for dim_name, emb in models.items():
            if w1 not in emb or w2 not in emb:
                row += f" {'N/A':>8} {'N/A':>10} │"
                valid = False
            else:
                v1, v2 = emb[w1], emb[w2]
                cos = cosine_similarity(v1, v2)
                dot = dot_product(v1, v2)
                results[dim_name]["cosine"].append(cos)
                results[dim_name]["dot"].append(dot)
                row += f" {cos:>8.4f} {dot:>10.2f} │"
        
        print(row)
    
    # Promedios
    print(f"{'-'*90}")
    avg_row = f"{'PROMEDIO':<25} │"
    for dim_name in models.keys():
        if results[dim_name]["cosine"]:
            avg_cos = sum(results[dim_name]["cosine"]) / len(results[dim_name]["cosine"])
            avg_dot = sum(results[dim_name]["dot"]) / len(results[dim_name]["dot"])
            avg_row += f" {avg_cos:>8.4f} {avg_dot:>10.2f} │"
        else:
            avg_row += f" {'N/A':>8} {'N/A':>10} │"
    print(avg_row)
    
    return results

# Ejecutar comparación para pares relacionados
related_results = compare_across_dimensions(related_pairs, glove_models, "PARES RELACIONADOS")

# Ejecutar comparación para pares no relacionados  
unrelated_results = compare_across_dimensions(unrelated_pairs, glove_models, "PARES NO RELACIONADOS")


  PARES RELACIONADOS
Palabra 1    Palabra 2    │  50d Cos    50d Dot │ 100d Cos   100d Dot │ 300d Cos   300d Dot
------------------------------------------------------------------------------------------
king         queen        │   0.7839      21.88 │   0.7508      27.59 │   0.6336      30.78 │
cat          dog          │   0.9218      19.74 │   0.8798      25.00 │   0.6817      28.78 │
happy        joyful       │   0.5550      11.04 │   0.5260      12.32 │   0.4751      15.16 │
car          vehicle      │   0.8834      27.46 │   0.8631      32.49 │   0.7655      35.86 │
doctor       nurse        │   0.7977      19.29 │   0.7522      22.12 │   0.5860      23.54 │
sun          moon         │   0.6543      17.12 │   0.6138      22.21 │   0.4807      24.72 │
coffee       tea          │   0.8080      20.95 │   0.7733      26.54 │   0.6692      32.34 │
france       paris        │   0.8025      26.57 │   0.7482      31.54 │   0.6581      34.27 │
-------------------------------------------

In [7]:
# -------------------------------------------------------
# 8. Resumen comparativo: Related vs Unrelated por dimensión
# -------------------------------------------------------

def print_summary(related: dict, unrelated: dict):
    """Imprime tabla resumen comparando related vs unrelated por dimensión."""
    
    print("\n" + "="*70)
    print("  RESUMEN: PROMEDIOS POR DIMENSIÓN")
    print("="*70)
    print(f"{'Dimensión':<12} │ {'Tipo':<12} │ {'Cosine Sim':>12} │ {'Dot Product':>14}")
    print("-"*70)
    
    for dim in ["50d", "100d", "300d"]:
        # Related
        if related[dim]["cosine"]:
            rel_cos = sum(related[dim]["cosine"]) / len(related[dim]["cosine"])
            rel_dot = sum(related[dim]["dot"]) / len(related[dim]["dot"])
        else:
            rel_cos, rel_dot = 0, 0
            
        # Unrelated
        if unrelated[dim]["cosine"]:
            unrel_cos = sum(unrelated[dim]["cosine"]) / len(unrelated[dim]["cosine"])
            unrel_dot = sum(unrelated[dim]["dot"]) / len(unrelated[dim]["dot"])
        else:
            unrel_cos, unrel_dot = 0, 0
        
        print(f"{dim:<12} │ {'Related':<12} │ {rel_cos:>12.4f} │ {rel_dot:>14.2f}")
        print(f"{'':<12} │ {'Unrelated':<12} │ {unrel_cos:>12.4f} │ {unrel_dot:>14.2f}")
        
        # Diferencia
        diff_cos = rel_cos - unrel_cos
        diff_dot = rel_dot - unrel_dot
        print(f"{'':<12} │ {'Δ (R - U)':<12} │ {diff_cos:>12.4f} │ {diff_dot:>14.2f}")
        print("-"*70)

print_summary(related_results, unrelated_results)

print("\n📊 INTERPRETACIÓN:")
print("   • Cosine Similarity: Mide la similitud angular (independiente de magnitud)")
print("   • Dot Product: Depende tanto del ángulo como de la magnitud de los vectores")
print("   • Δ (R - U): Diferencia entre pares relacionados y no relacionados")
print("   • Mayor Δ = mejor separación semántica entre palabras relacionadas y no relacionadas")


  RESUMEN: PROMEDIOS POR DIMENSIÓN
Dimensión    │ Tipo         │   Cosine Sim │    Dot Product
----------------------------------------------------------------------
50d          │ Related      │       0.7758 │          20.51
             │ Unrelated    │       0.1529 │           4.08
             │ Δ (R - U)    │       0.6229 │          16.43
----------------------------------------------------------------------
100d         │ Related      │       0.7384 │          24.98
             │ Unrelated    │       0.1175 │           3.85
             │ Δ (R - U)    │       0.6209 │          21.13
----------------------------------------------------------------------
300d         │ Related      │       0.6187 │          28.18
             │ Unrelated    │       0.0470 │           2.20
             │ Δ (R - U)    │       0.5717 │          25.99
----------------------------------------------------------------------

📊 INTERPRETACIÓN:
   • Cosine Similarity: Mide la similitud angular (independie

In [8]:
# -------------------------------------------------------
# 9. COMPARACIÓN COMPLETA: Analogías con Cosine y Dot Product
# -------------------------------------------------------

def most_similar_with_both_metrics(v: torch.Tensor, emb: dict, top_k: int = 10, metric: str = "cosine") -> list[tuple[str, float, float]]:
    """
    Encuentra las palabras más similares a un vector.
    Retorna: lista de (palabra, cosine_sim, dot_product)
    """
    words = list(emb.keys())
    vecs = torch.stack([emb[w] for w in words])
    
    v_norm = v / v.norm()
    vecs_norm = vecs / vecs.norm(dim=1, keepdim=True)
    
    cosine_sims = torch.matmul(vecs_norm, v_norm)
    dot_products = torch.matmul(vecs, v)
    
    # Ordenar por la métrica elegida
    if metric == "cosine":
        topk = torch.topk(cosine_sims, top_k)
    else:
        topk = torch.topk(dot_products, top_k)
    
    results = []
    for j, idx in enumerate(topk.indices):
        word = words[idx]
        cos = float(cosine_sims[idx])
        dot = float(dot_products[idx])
        results.append((word, cos, dot))
    
    return results


def run_analogy_both_metrics(a: str, b: str, c: str, emb: dict, target: str, top_k: int = 10, metric: str = "cosine") -> dict:
    """
    Ejecuta analogía: a - b + c = ?
    Retorna info con ambas métricas.
    """
    if a not in emb or b not in emb or c not in emb or target not in emb:
        return None
    
    result_vec = emb[a] - emb[b] + emb[c]
    neighbors = most_similar_with_both_metrics(result_vec, emb, top_k, metric)
    
    # Encontrar posición del target
    rank = None
    target_cos = None
    target_dot = None
    for i, (word, cos, dot) in enumerate(neighbors):
        if word == target:
            rank = i + 1
            target_cos = cos
            target_dot = dot
            break
    
    # Si target no está en top_k, calcular sus métricas directamente
    if rank is None:
        target_vec = emb[target]
        v_norm = result_vec / result_vec.norm()
        t_norm = target_vec / target_vec.norm()
        target_cos = float(torch.dot(v_norm, t_norm))
        target_dot = float(torch.dot(result_vec, target_vec))
    
    return {
        "top1_word": neighbors[0][0],
        "top1_cos": neighbors[0][1],
        "top1_dot": neighbors[0][2],
        "target_rank": rank,
        "target_cos": target_cos,
        "target_dot": target_dot,
        "neighbors": neighbors
    }


# Definir analogías a probar
analogies = [
    {"a": "king", "b": "man", "c": "woman", "target": "queen", "name": "king - man + woman"},
    {"a": "colder", "b": "cold", "c": "hot", "target": "hotter", "name": "colder - cold + hot"},
    {"a": "paris", "b": "france", "c": "spain", "target": "madrid", "name": "paris - france + spain"},
    {"a": "bigger", "b": "big", "c": "small", "target": "smaller", "name": "bigger - big + small"},
]

# ============================================
# TABLA 1: Rankings por COSINE SIMILARITY
# ============================================
print("=" * 115)
print("  ANALOGÍAS ORDENADAS POR COSINE SIMILARITY")
print("=" * 115)
print(f"\n{'Analogía':<25} │ {'Dim':^6} │ {'Target':^8} │ {'Rank':^5} │ {'Cos':^7} │ {'Dot':^9} │ {'Top1':^10} │ {'Top1 Cos':^8} │ {'Top1 Dot':^9}")
print("-" * 115)

for analogy in analogies:
    for i, (dim_name, emb) in enumerate(glove_models.items()):
        result = run_analogy_both_metrics(analogy["a"], analogy["b"], analogy["c"], emb, analogy["target"], metric="cosine")
        
        if result is None:
            continue
        
        name = analogy["name"] if i == 0 else ""
        rank_str = str(result["target_rank"]) if result["target_rank"] else ">10"
        
        print(f"{name:<25} │ {dim_name:^6} │ {analogy['target']:^8} │ {rank_str:^5} │ {result['target_cos']:^7.4f} │ {result['target_dot']:^9.2f} │ {result['top1_word']:^10} │ {result['top1_cos']:^8.4f} │ {result['top1_dot']:^9.2f}")
    
    print("-" * 115)

# ============================================
# TABLA 2: Rankings por DOT PRODUCT
# ============================================
print("\n" + "=" * 115)
print("  ANALOGÍAS ORDENADAS POR DOT PRODUCT")
print("=" * 115)
print(f"\n{'Analogía':<25} │ {'Dim':^6} │ {'Target':^8} │ {'Rank':^5} │ {'Cos':^7} │ {'Dot':^9} │ {'Top1':^10} │ {'Top1 Cos':^8} │ {'Top1 Dot':^9}")
print("-" * 115)

for analogy in analogies:
    for i, (dim_name, emb) in enumerate(glove_models.items()):
        result = run_analogy_both_metrics(analogy["a"], analogy["b"], analogy["c"], emb, analogy["target"], metric="dot")
        
        if result is None:
            continue
        
        name = analogy["name"] if i == 0 else ""
        rank_str = str(result["target_rank"]) if result["target_rank"] else ">10"
        
        print(f"{name:<25} │ {dim_name:^6} │ {analogy['target']:^8} │ {rank_str:^5} │ {result['target_cos']:^7.4f} │ {result['target_dot']:^9.2f} │ {result['top1_word']:^10} │ {result['top1_cos']:^8.4f} │ {result['top1_dot']:^9.2f}")
    
    print("-" * 115)

  ANALOGÍAS ORDENADAS POR COSINE SIMILARITY

Analogía                  │  Dim   │  Target  │ Rank  │   Cos   │    Dot    │    Top1    │ Top1 Cos │ Top1 Dot 
-------------------------------------------------------------------------------------------------------------------
king - man + woman        │  50d   │  queen   │   2   │ 0.8610  │   24.64   │    king    │  0.8860  │   26.24  
                          │  100d  │  queen   │   2   │ 0.7834  │   29.91   │    king    │  0.8552  │   33.25  
                          │  300d  │  queen   │   2   │ 0.6896  │   38.17   │    king    │  0.8066  │   45.13  
-------------------------------------------------------------------------------------------------------------------


colder - cold + hot       │  50d   │  hotter  │   3   │ 0.8125  │   22.55   │   colder   │  0.8501  │   28.27  
                          │  100d  │  hotter  │   2   │ 0.7616  │   28.47   │   colder   │  0.7812  │   31.76  
                          │  300d  │  hotter  │   1   │ 0.7156  │   41.49   │   hotter   │  0.7156  │   41.49  
-------------------------------------------------------------------------------------------------------------------
paris - france + spain    │  50d   │  madrid  │   3   │ 0.8017  │   23.49   │   aires    │  0.8462  │   21.64  
                          │  100d  │  madrid  │   1   │ 0.7962  │   29.49   │   madrid   │  0.7962  │   29.49  
                          │  300d  │  madrid  │   1   │ 0.7379  │   39.67   │   madrid   │  0.7379  │   39.67  
-------------------------------------------------------------------------------------------------------------------
bigger - big + small      │  50d   │ smaller  │   4   │ 0.8655  │   22.29   │   larger   │  0.90

In [9]:
# -------------------------------------------------------
# 11. TABLAS EN FORMATO MARKDOWN (para copiar)
# -------------------------------------------------------

def generate_markdown_tables():
    """Genera todas las tablas en formato markdown."""
    
    output = []
    
    # ============================================
    # TABLA 1: Analogías con Rank Cosine y Rank Dot
    # ============================================
    output.append("## Analogías por dimensión (Cosine y Dot Product)\n")
    output.append("| Analogía | Dim | Target | Rank Cos | Rank Dot | Cosine | Dot |")
    output.append("|:---|:---:|:---:|:---:|:---:|:---:|:---:|")
    
    for analogy in analogies:
        first_row = True
        for dim_name, emb in glove_models.items():
            # Obtener resultados con ambos rankings
            result_cos = run_analogy_both_metrics(analogy["a"], analogy["b"], analogy["c"], emb, analogy["target"], metric="cosine")
            result_dot = run_analogy_both_metrics(analogy["a"], analogy["b"], analogy["c"], emb, analogy["target"], metric="dot")
            
            if result_cos is None:
                continue
            
            name = analogy["name"] if first_row else ""
            rank_cos_str = str(result_cos["target_rank"]) if result_cos["target_rank"] else ">10"
            rank_dot_str = str(result_dot["target_rank"]) if result_dot["target_rank"] else ">10"
            
            output.append(f"| {name} | {dim_name} | {analogy['target']} | {rank_cos_str} | {rank_dot_str} | {result_cos['target_cos']:.4f} | {result_cos['target_dot']:.2f} |")
            first_row = False
    
    # ============================================
    # TABLA 2: Pares Relacionados
    # ============================================
    output.append("\n## Pares Relacionados\n")
    output.append("| Palabra 1 | Palabra 2 | 50d Cos | 50d Dot | 100d Cos | 100d Dot | 300d Cos | 300d Dot |")
    output.append("|:---|:---|:---:|:---:|:---:|:---:|:---:|:---:|")
    
    totals = {dim: {"cos": [], "dot": []} for dim in glove_models.keys()}
    
    for w1, w2 in related_pairs:
        row = f"| {w1} | {w2} |"
        for dim_name, emb in glove_models.items():
            if w1 in emb and w2 in emb:
                cos = cosine_similarity(emb[w1], emb[w2])
                dot = dot_product(emb[w1], emb[w2])
                totals[dim_name]["cos"].append(cos)
                totals[dim_name]["dot"].append(dot)
                row += f" {cos:.4f} | {dot:.2f} |"
            else:
                row += " N/A | N/A |"
        output.append(row)
    
    # Promedio
    row = "| **PROMEDIO** | |"
    for dim_name in glove_models.keys():
        if totals[dim_name]["cos"]:
            avg_cos = sum(totals[dim_name]["cos"]) / len(totals[dim_name]["cos"])
            avg_dot = sum(totals[dim_name]["dot"]) / len(totals[dim_name]["dot"])
            row += f" **{avg_cos:.4f}** | **{avg_dot:.2f}** |"
        else:
            row += " N/A | N/A |"
    output.append(row)
    
    # ============================================
    # TABLA 3: Pares No Relacionados
    # ============================================
    output.append("\n## Pares No Relacionados\n")
    output.append("| Palabra 1 | Palabra 2 | 50d Cos | 50d Dot | 100d Cos | 100d Dot | 300d Cos | 300d Dot |")
    output.append("|:---|:---|:---:|:---:|:---:|:---:|:---:|:---:|")
    
    totals = {dim: {"cos": [], "dot": []} for dim in glove_models.keys()}
    
    for w1, w2 in unrelated_pairs:
        row = f"| {w1} | {w2} |"
        for dim_name, emb in glove_models.items():
            if w1 in emb and w2 in emb:
                cos = cosine_similarity(emb[w1], emb[w2])
                dot = dot_product(emb[w1], emb[w2])
                totals[dim_name]["cos"].append(cos)
                totals[dim_name]["dot"].append(dot)
                row += f" {cos:.4f} | {dot:.2f} |"
            else:
                row += " N/A | N/A |"
        output.append(row)
    
    # Promedio
    row = "| **PROMEDIO** | |"
    for dim_name in glove_models.keys():
        if totals[dim_name]["cos"]:
            avg_cos = sum(totals[dim_name]["cos"]) / len(totals[dim_name]["cos"])
            avg_dot = sum(totals[dim_name]["dot"]) / len(totals[dim_name]["dot"])
            row += f" **{avg_cos:.4f}** | **{avg_dot:.2f}** |"
        else:
            row += " N/A | N/A |"
    output.append(row)
    
    return "\n".join(output)

# Generar e imprimir markdown
markdown_output = generate_markdown_tables()
print(markdown_output)

## Analogías por dimensión (Cosine y Dot Product)

| Analogía | Dim | Target | Rank Cos | Rank Dot | Cosine | Dot |
|:---|:---:|:---:|:---:|:---:|:---:|:---:|
| king - man + woman | 50d | queen | 2 | 3 | 0.8610 | 24.64 |
|  | 100d | queen | 2 | 2 | 0.7834 | 29.91 |
|  | 300d | queen | 2 | 2 | 0.6896 | 38.17 |
| colder - cold + hot | 50d | hotter | 3 | >10 | 0.8125 | 22.55 |
|  | 100d | hotter | 2 | 2 | 0.7616 | 28.47 |
|  | 300d | hotter | 1 | 1 | 0.7156 | 41.49 |
| paris - france + spain | 50d | madrid | 3 | 3 | 0.8017 | 23.49 |
|  | 100d | madrid | 1 | 1 | 0.7962 | 29.49 |
|  | 300d | madrid | 1 | 1 | 0.7379 | 39.67 |
| bigger - big + small | 50d | smaller | 4 | 3 | 0.8655 | 22.29 |
|  | 100d | smaller | 2 | 2 | 0.8575 | 26.74 |
|  | 300d | smaller | 2 | 1 | 0.7753 | 31.71 |

## Pares Relacionados

| Palabra 1 | Palabra 2 | 50d Cos | 50d Dot | 100d Cos | 100d Dot | 300d Cos | 300d Dot |
|:---|:---|:---:|:---:|:---:|:---:|:---:|:---:|
| king | queen | 0.7839 | 21.88 | 0.7508 | 27.59 |

In [10]:
# -------------------------------------------------------
# Top 10 de (colder - cold + hot) en 50d ordenado por Dot Product
# -------------------------------------------------------

result_vec_50d = glove_50d["colder"] - glove_50d["cold"] + glove_50d["hot"]
neighbors = most_similar_with_both_metrics(result_vec_50d, glove_50d, top_k=10, metric="dot")

print("Top 10 de (colder - cold + hot) en 50d - ordenado por DOT PRODUCT:\n")
print(f"{'Rank':<6} {'Palabra':<15} {'Cosine':>10} {'Dot Product':>12}")
print("-" * 45)

for i, (word, cos, dot) in enumerate(neighbors, 1):
    marker = " ← hotter" if word == "hotter" else ""
    print(f"{i:<6} {word:<15} {cos:>10.4f} {dot:>12.2f}{marker}")

Top 10 de (colder - cold + hot) en 50d - ordenado por DOT PRODUCT:

Rank   Palabra             Cosine  Dot Product
---------------------------------------------
1      colder              0.8501        28.27
2      petruno             0.5171        27.90
3      temperatures        0.7281        27.55
4      music/club          0.4532        27.04
5      seasonably          0.6235        26.26
6      drier               0.7648        24.88
7      precipitation       0.6816        24.79
8      viser               0.5175        24.57
9      cooler              0.8142        24.24
10     humid               0.6629        23.49
